In [1]:
import csv
from anarcii import Anarcii
from Bio import SeqIO
import pathlib, sys

In [2]:
import torch
import pandas as pd
import glob
from pathlib import Path
print(torch.cuda.is_available())   # → True

True


In [9]:
from pathlib import Path
from anarcii import Anarcii
from itertools import islice
import csv
import time
import os
import pandas as pd
import numpy as np

# ================== SETTINGS ==================
fasta_path = Path(r"D:\Thesis - November\Dataset\nanobodies.fasta")
long_csv   = fasta_path.with_stem(fasta_path.stem + "_imgt_long").with_suffix(".csv")
wide_csv   = fasta_path.with_stem(fasta_path.stem + "_imgt_wide").with_suffix(".csv")

CHUNK      = 5000
USE_GPU    = True
MODE       = "speed"

# ================= FASTA HELPERS ===============
def fasta_iter(fp: Path):
    """Yield (name, sequence) from FASTA."""
    head = None
    seq = []
    with fp.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if head:
                    yield head, "".join(seq)
                head = line[1:]
                seq = []
            else:
                seq.append(line)
        if head:
            yield head, "".join(seq)

def batched(it, n):
    it = iter(it)
    while True:
        batch = list(islice(it, n))
        if not batch:
            break
        yield batch

# ============== STEP 1: RUN ANARCII → LONG CSV ==========
print(f"Loading Anarcii (GPU={USE_GPU})...")
model = Anarcii(seq_type="antibody", mode=MODE, cpu=not USE_GPU)
print(f"Using device: {model.device}")

# Header for long-format file
with long_csv.open("w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow([
        "Name", "Chain", "Score", "Query_start", "Query_end",
        "pos", "insertion", "aa", "error"
    ])

total = sum(1 for _ in fasta_iter(fasta_path))
total_chunks = (total + CHUNK - 1) // CHUNK
print(f"Total sequences: {total:,} | chunks: {total_chunks}")

start = time.time()
chunk_i = 0

for batch in batched(fasta_iter(fasta_path), CHUNK):
    chunk_i += 1
    t0 = time.time()

    # Temporary FASTA for this chunk
    tmp_fa = fasta_path.parent / f"_tmp_{chunk_i}.fasta"
    with tmp_fa.open("w", encoding="utf-8") as f:
        for name, seq in batch:
            f.write(f">{name}\n{seq}\n")

    # Run numbering on this chunk
    results = model.number(str(tmp_fa))

    # Append to long CSV
    with long_csv.open("a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)

        for name, info in results.items():
            chain = info.get("chain_type", "")
            score = info.get("score", "")
            qs    = info.get("query_start", "")
            qe    = info.get("query_end", "")
            err   = info.get("error")
            numbering = info.get("numbering")

            # If numbering failed, keep one row with empty pos/insertion/aa
            if numbering is None:
                w.writerow([name, chain, score, qs, qe,
                            "", "", "", err])
            else:
                for ((pos, ins), aa) in numbering:
                    ins = (ins or " ").strip()   # '' means no insertion
                    w.writerow([name, chain, score, qs, qe,
                                pos, ins, aa, err])

    # Clean up temp file
    try:
        os.remove(tmp_fa)
    except OSError:
        pass

    dt = time.time() - t0
    pct = chunk_i / total_chunks * 100
    eta = (total_chunks - chunk_i) * dt
    print(f"[chunk {chunk_i}/{total_chunks}] {len(batch)} seqs | "
          f"{dt:.1f}s | {pct:.1f}% | ETA ~ {eta/60:.1f} min")

print(f"Step 1 done: long CSV written to {long_csv}")
print(f"Elapsed: {(time.time() - start)/60:.1f} min")

# ============== STEP 2: LONG → WIDE (DYNAMIC IMGT) ==========

print("\nStep 2: building wide IMGT table from long CSV...")

df = pd.read_csv(long_csv)

# Ensure numeric pos where present
df["pos"] = pd.to_numeric(df["pos"], errors="coerce")

# Meta information per sequence (all sequences, even failed)
meta_cols = ["Name", "Chain", "Score", "Query_start", "Query_end"]
meta = (
    df[meta_cols]
    .drop_duplicates(subset=["Name"])
    .set_index("Name")
)

# Only rows with valid positions for building AA matrix
df_valid = df[df["pos"].notna()].copy()

# Build labels like "32", "32A", "69B" directly from Anarcii output
def make_label(row):
    pos = int(row["pos"])
    ins = str(row["insertion"]) if pd.notnull(row["insertion"]) else ""
    ins = ins.strip()
    return f"{pos}{ins}"

df_valid["pos_label"] = df_valid.apply(make_label, axis=1)

# All unique position labels from actual numbering (dynamic IMGT columns)
labels = sorted(
    df_valid["pos_label"].unique(),
    key=lambda s: (
        int("".join(ch for ch in s if ch.isdigit())),
        "".join(ch for ch in s if ch.isalpha())
    )
)

# Pivot: Name × pos_label → aa
wide_aa = (
    df_valid
    .pivot_table(index="Name", columns="pos_label", values="aa",
                 aggfunc="first")
)

# Ensure all labels exist as columns in AA matrix
for lab in labels:
    if lab not in wide_aa.columns:
        wide_aa[lab] = np.nan
wide_aa = wide_aa[labels]

# Join meta info (includes sequences that had no numbering)
wide = meta.join(wide_aa, how="left")

# Fill amino-acid cells with '-' where missing
for lab in labels:
    if lab in wide.columns:
        wide[lab] = wide[lab].fillna("-")

# Convert column names to match your style
wide = wide.reset_index()
wide = wide.rename(columns={
    "Query_start": "Query start",
    "Query_end": "Query end"
})

# Final column order: Name, Chain, Score, Query start, Query end, then IMGT labels
final_cols = ["Name", "Chain", "Score", "Query start", "Query end"] + labels
wide = wide[final_cols]

wide.to_csv(wide_csv, index=False)
print(f"Step 2 done: wide IMGT CSV written to {wide_csv}")


Loading Anarcii (GPU=True)...
Using device CUDA with 12 CPUs
Using device: cuda
Total sequences: 149,006 | chunks: 30
[chunk 1/30] 5000 seqs | 260.8s | 3.3% | ETA ~ 126.1 min
[chunk 2/30] 5000 seqs | 175.1s | 6.7% | ETA ~ 81.7 min
[chunk 3/30] 5000 seqs | 91.9s | 10.0% | ETA ~ 41.3 min
[chunk 4/30] 5000 seqs | 94.4s | 13.3% | ETA ~ 40.9 min
[chunk 5/30] 5000 seqs | 110.3s | 16.7% | ETA ~ 46.0 min
[chunk 6/30] 5000 seqs | 102.2s | 20.0% | ETA ~ 40.9 min
[chunk 7/30] 5000 seqs | 108.8s | 23.3% | ETA ~ 41.7 min
[chunk 8/30] 5000 seqs | 106.9s | 26.7% | ETA ~ 39.2 min
[chunk 9/30] 5000 seqs | 107.6s | 30.0% | ETA ~ 37.6 min
[chunk 10/30] 5000 seqs | 110.6s | 33.3% | ETA ~ 36.9 min
[chunk 11/30] 5000 seqs | 107.4s | 36.7% | ETA ~ 34.0 min
[chunk 12/30] 5000 seqs | 107.3s | 40.0% | ETA ~ 32.2 min
[chunk 13/30] 5000 seqs | 103.2s | 43.3% | ETA ~ 29.2 min
[chunk 14/30] 5000 seqs | 127.8s | 46.7% | ETA ~ 34.1 min
[chunk 15/30] 5000 seqs | 120.0s | 50.0% | ETA ~ 30.0 min
[chunk 16/30] 5000 seqs 

C:\Users\ankit\AppData\Local\Temp\ipykernel_18428\3121022345.py:119: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(long_csv)


Step 2 done: wide IMGT CSV written to D:\Thesis - November\Dataset\nanobodies_imgt_wide.csv


Fix column insertions

In [ ]:
import pandas as pd
import numpy as np

# --------------------
# Load your existing long-format file
# --------------------
long_csv = r"D:\Thesis - November\Dataset\nanobodies_imgt_long.csv"
wide_csv = long_csv.replace("_long", "_wide_fixed")

df = pd.read_csv(long_csv)

# Ensure numeric pos where possible
df["pos"] = pd.to_numeric(df["pos"], errors="coerce")

# --------------------
# Build IMGT labels for pivot
# --------------------
def make_label(row):
    if pd.isna(row["pos"]):
        return None
    pos = int(row["pos"])
    ins = "" if pd.isna(row["insertion"]) else str(row["insertion"]).strip()
    return f"{pos}{ins}"

df["pos_label"] = df.apply(make_label, axis=1)

# Keep only valid labels
df_valid = df[df["pos_label"].notna()].copy()

# --------------------
# Custom IMGT ordering (111 and 112 special)
# --------------------
def parse_label(label):
    digits = "".join(ch for ch in label if ch.isdigit())
    letters = "".join(ch for ch in label if ch.isalpha())
    return int(digits), letters

def imgt_sort(label):
    base, ins = parse_label(label)

    # --- Position 111: 111,111A..111I ---
    if base == 111:
        if ins == "":
            order = 0
        else:
            order = "ABCDEFGHI".find(ins) + 1
        return (111, order)

    # --- Position 112: 112I..112A, then 112 ---
    if base == 112:
        if ins == "":
            return (112, 50)  # after all insertions
        order = "IHGFEDCBA".find(ins) + 1
        return (112, order)

    # --- Default ordering: base, then A,B,C... ---
    if ins == "":
        return (base, 0)
    return (base, ord(ins) - ord("A") + 1)

# Build the final set of labels seen
labels = sorted(
    df_valid["pos_label"].unique(),
    key=imgt_sort
)

# --------------------
# Build wide pivot table
# --------------------
wide = (
    df_valid
    .pivot_table(index="Name", columns="pos_label", values="aa", aggfunc="first")
)

# Ensure missing columns from failed sequences
for lab in labels:
    if lab not in wide.columns:
        wide[lab] = np.nan

wide = wide[labels].fillna("-")

# --------------------
# Merge meta info
# --------------------
meta = df[["Name", "Chain", "Score", "Query_start", "Query_end"]].drop_duplicates("Name")
meta = meta.set_index("Name")
wide = meta.join(wide, how="left").reset_index()

# Fix column names
wide = wide.rename(columns={
    "Query_start": "Query start",
    "Query_end": "Query end"
})

# --------------------
# Save final IMGT-wide file
# --------------------
wide.to_csv(wide_csv, index=False)
print("Written fixed IMGT wide file to:", wide_csv)


C:\Users\ankit\AppData\Local\Temp\ipykernel_22996\1462224731.py:10: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(long_csv)


Written fixed IMGT wide file to: D:\Thesis - November\Dataset\nanobodies_imgt_wide_fixed.csv.csv


In [4]:
from anarcii import Anarcii
model = Anarcii(seq_type="antibody", mode="speed", cpu=False)

test = """>test
QVQLQESGGGLVQAGGSLRLSCAASGSISGDGDMGWYRQAPGKERELVASIARGGSTNYADSVKGRFTISRDNAKNTVYLQMNSLKPEDTAVYYCAAHAVLGYDHDYWGQGTQVTVSS
"""

with open("test.fa","w") as f:
    f.write(test)

result = model.number("test.fa")
print(result)


Using device CUDA with 12 CPUs
{'test': {'numbering': [((1, ' '), 'Q'), ((2, ' '), 'V'), ((3, ' '), 'Q'), ((4, ' '), 'L'), ((5, ' '), 'Q'), ((6, ' '), 'E'), ((7, ' '), 'S'), ((8, ' '), 'G'), ((9, ' '), 'G'), ((10, ' '), '-'), ((11, ' '), 'G'), ((12, ' '), 'L'), ((13, ' '), 'V'), ((14, ' '), 'Q'), ((15, ' '), 'A'), ((16, ' '), 'G'), ((17, ' '), 'G'), ((18, ' '), 'S'), ((19, ' '), 'L'), ((20, ' '), 'R'), ((21, ' '), 'L'), ((22, ' '), 'S'), ((23, ' '), 'C'), ((24, ' '), 'A'), ((25, ' '), 'A'), ((26, ' '), 'S'), ((27, ' '), 'G'), ((28, ' '), 'S'), ((29, ' '), 'I'), ((30, ' '), 'S'), ((31, ' '), '-'), ((32, ' '), '-'), ((33, ' '), '-'), ((34, ' '), '-'), ((35, ' '), 'G'), ((36, ' '), 'D'), ((37, ' '), 'G'), ((38, ' '), 'D'), ((39, ' '), 'M'), ((40, ' '), 'G'), ((41, ' '), 'W'), ((42, ' '), 'Y'), ((43, ' '), 'R'), ((44, ' '), 'Q'), ((45, ' '), 'A'), ((46, ' '), 'P'), ((47, ' '), 'G'), ((48, ' '), 'K'), ((49, ' '), 'E'), ((50, ' '), 'R'), ((51, ' '), 'E'), ((52, ' '), 'L'), ((53, ' '), 'V'), 